# Implementasi Bab 13 — CNN dan Transfer Learning

Transfer learning MobileNetV2 dan CNN dari nol [ML-MODUL-01, hlm. 86–88]. Adaptasi utama: contoh folder citra diganti CIFAR-10 bawaan Keras agar notebook mandiri. Runtime pertama dapat mengunduh dataset dan bobot ImageNet.


In [ ]:
import random
import time
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__)


In [ ]:
(X_train_all, y_train_all), (X_test_all, y_test_all) = keras.datasets.cifar10.load_data()
y_train_all = y_train_all.ravel()
y_test_all = y_test_all.ravel()

rng = np.random.default_rng(SEED)
train_indices = rng.choice(len(X_train_all), size=5000, replace=False)
test_indices = rng.choice(len(X_test_all), size=1000, replace=False)
X_train, y_train = X_train_all[train_indices], y_train_all[train_indices]
X_test, y_test = X_test_all[test_indices], y_test_all[test_indices]
print("Subset train/test:", X_train.shape, X_test.shape)


In [ ]:
class_names = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for index, axis in enumerate(axes.ravel()):
    axis.imshow(X_train[index])
    axis.set_title(class_names[y_train[index]])
    axis.axis("off")
plt.tight_layout()
plt.show()


## Transfer learning MobileNetV2

Citra CIFAR-10 berukuran 32×32 di-resize menjadi 96×96. `preprocess_input` mengikuti normalisasi MobileNetV2; base model dibekukan untuk feature extraction [ML-MODUL-01, hlm. 86–87].


In [ ]:
image_size = (96, 96)
base_model = MobileNetV2(
    input_shape=image_size + (3,), include_top=False, weights="imagenet"
)
base_model.trainable = False

inputs = keras.Input(shape=(32, 32, 3))
x = keras.layers.Resizing(*image_size)(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dense(128, activation="relu")(x)
x = keras.layers.Dropout(0.3)(x)
outputs = keras.layers.Dense(10, activation="softmax")(x)
transfer_model = keras.Model(inputs, outputs, name="mobilenetv2_cifar10")
transfer_model.compile(
    optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)
transfer_model.summary()


In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=2, restore_best_weights=True
)
start = time.perf_counter()
transfer_history = transfer_model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1,
)
transfer_seconds = time.perf_counter() - start
transfer_loss, transfer_accuracy = transfer_model.evaluate(X_test, y_test, verbose=0)
print(f"Transfer learning accuracy={transfer_accuracy:.3f}, waktu={transfer_seconds:.1f}s")


In [ ]:
plt.plot(transfer_history.history["accuracy"], label="Training Accuracy")
plt.plot(transfer_history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Akurasi")
plt.title("MobileNetV2 Feature Extraction")
plt.legend()
plt.show()


## Latihan: CNN sederhana dari nol

Arsitektur mengikuti latihan modul dan memakai subset yang sama agar perbandingan waktu serta akurasi lebih adil [ML-MODUL-01, hlm. 87–88].


In [ ]:
tf.keras.utils.set_random_seed(SEED)
cnn_model = keras.Sequential([
    keras.layers.Input(shape=(32, 32, 3)),
    keras.layers.Rescaling(1.0 / 255),
    keras.layers.Conv2D(32, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])
cnn_model.compile(
    optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)
start = time.perf_counter()
cnn_history = cnn_model.fit(
    X_train, y_train, validation_split=0.2, epochs=5, batch_size=64, verbose=1
)
cnn_seconds = time.perf_counter() - start
cnn_loss, cnn_accuracy = cnn_model.evaluate(X_test, y_test, verbose=0)
print(f"CNN scratch accuracy={cnn_accuracy:.3f}, waktu={cnn_seconds:.1f}s")


In [ ]:
comparison = {
    "Transfer Learning": {
        "test_accuracy": transfer_accuracy,
        "training_seconds": transfer_seconds,
        "trainable_parameters": int(np.sum([np.prod(weight.shape) for weight in transfer_model.trainable_weights])),
    },
    "CNN Scratch": {
        "test_accuracy": cnn_accuracy,
        "training_seconds": cnn_seconds,
        "trainable_parameters": int(np.sum([np.prod(weight.shape) for weight in cnn_model.trainable_weights])),
    },
}
comparison


## Kesimpulan

Transfer learning memanfaatkan fitur ImageNet dan hanya melatih classifier baru saat base dibekukan. CNN scratch lebih bebas menyesuaikan domain tetapi biasanya memerlukan lebih banyak data/epoch. Hasil subset kecil adalah demonstrasi akademik, bukan benchmark final.
